In [1]:
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os


In [14]:
# class RESNET50(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
#         num_features = self.model.fc.in_features
#         self.model.fc = nn.Linear(num_features, 2)

#     def forward(self, x):
#         return self.model(x)
import torch.nn as nn
import torchvision.models as models
from torchvision.models.resnet import BasicBlock, Bottleneck

class RESNET50(models.ResNet):
    def __init__(self):
        super().__init__(block=models.Bottleneck, layers=[3,4,6,3])
        pretrained = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.load_state_dict(pretrained.state_dict(), strict=False)

        num_features = self.fc.in_features
        self.fc = nn.Linear(num_features, 2)  # output 2 class

In [15]:
# class RESNET18(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
#         num_features = self.model.fc.in_features
#         self.model.fc = nn.Linear(num_features, 2)

#     def forward(self, x):
#         return self.model(x)
class RESNET18(models.ResNet):
    def __init__(self):
        super().__init__(block=models.BasicBlock, layers=[2,2,2,2])
        # Load pretrained weights, bỏ qua phần fully connected cuối
        pretrained = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.load_state_dict(pretrained.state_dict(), strict=False)

        num_features = self.fc.in_features
        self.fc = nn.Linear(num_features, 2)  # output 2 class

In [18]:
class RESNET18(models.ResNet):
    def __init__(self):
        super().__init__(block=BasicBlock, layers=[2, 2, 2, 2])
        pretrained = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.load_state_dict(pretrained.state_dict(), strict=False)
        num_features = self.fc.in_features
        self.fc = nn.Linear(num_features, 2)

class RESNET50(models.ResNet):
    def __init__(self):
        super().__init__(block=Bottleneck, layers=[3, 4, 6, 3])
        pretrained = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.load_state_dict(pretrained.state_dict(), strict=False)
        num_features = self.fc.in_features
        self.fc = nn.Linear(num_features, 2)

In [4]:
class DistillLoss(nn.Module):
    def __init__(self, T=4.0, alpha=0.7):
        super().__init__()
        self.T = T
        self.alpha = alpha
        self.kld = nn.KLDivLoss(reduction='batchmean')
        self.ce = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, true_labels):
        kd = self.kld(F.log_softmax(student_logits / self.T, dim=1),
                      F.softmax(teacher_logits / self.T, dim=1)) * (self.T ** 2)
        ce = self.ce(student_logits, true_labels)
        return self.alpha * kd + (1 - self.alpha) * ce

In [5]:
from torchvision import datasets

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/kaggle/input/fakeface-train-data-v3", transform=transform)
val_dataset   = datasets.ImageFolder("/kaggle/input/fakeface-valid-data-v2", transform=transform)
test_dataset = datasets.ImageFolder("/kaggle/input/fakeface-test-data-v2", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False,num_workers=2)

In [16]:
# import torch
# from collections import OrderedDict

# def load_teacher_model(path, device="cuda"):
#     model = RESNET50()
#     state_dict = torch.load(path, map_location=device)

#     new_state_dict = OrderedDict((k.replace("module.", ""), v) for k, v in state_dict.items())
#     model.load_state_dict(new_state_dict)

#     model.to(device)
#     model.eval()
#     return model
from collections import OrderedDict
import torch

def load_teacher_model(path, model_class, device):
    model = model_class()
    checkpoint = torch.load(path, map_location=device)

    # Nếu checkpoint có 'state_dict' bên trong thì lấy ra
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        # Xóa prefix 'module.' nếu có (do DataParallel)
        if k.startswith("module."):
            name = k[7:]
        else:
            name = k
        new_state_dict[name] = v

    model.load_state_dict(new_state_dict)
    model.to(device)
    model.eval()
    return model


In [19]:
# teacher = load_teacher_model("/kaggle/input/resnet50-teacher/resnet50_teacher.pth", device="cuda") # doi link nha
# student = RESNET18()

# if torch.cuda.device_count() > 1:
#     print(f"Sử dụng {torch.cuda.device_count()} GPU với DataParallel.")
#     student = nn.DataParallel(student)
#     teacher = nn.DataParallel(teacher)

# kd_loss_fn = DistillLoss(T=4.0, alpha=0.7)
# optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

teacher = load_teacher_model("/kaggle/input/resnet50-teacher/resnet50_teacher.pth", RESNET50, device)
student = RESNET18()
student.to(device)

if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPU với DataParallel.")
    student = nn.DataParallel(student)
    teacher = nn.DataParallel(teacher)

kd_loss_fn = DistillLoss(T=4.0, alpha=0.7)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s] 


Sử dụng 2 GPU với DataParallel.


In [20]:
student = student.to(device)
teacher = teacher.to(device)

In [21]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

def evaluate_model_on_validation(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = float('nan')

    return acc, auc

In [22]:
import time
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

def train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10):
    ce_loss_fn = nn.CrossEntropyLoss()
    start_training = time.time()

    for epoch in range(epochs):
        student.train()
        teacher.eval()

        total_kd_loss = 0
        total_student_loss = 0
        total_teacher_loss = 0

        all_probs = []
        all_labels = []

        start_epoch = time.time()

        progress_bar = tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training", leave=False)

        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                t_logits = teacher(x)
                teacher_ce_loss = ce_loss_fn(t_logits, y)

            s_logits = student(x)

            student_ce_loss = ce_loss_fn(s_logits, y)
            kd_loss = kd_loss_fn(s_logits, t_logits, y)

            kd_loss.backward()
            optimizer.step()

            total_kd_loss += kd_loss.item()
            total_student_loss += student_ce_loss.item()
            total_teacher_loss += teacher_ce_loss.item()

            probs = torch.softmax(s_logits, dim=1)[:, 1].detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y.cpu().numpy())

            progress_bar.set_postfix({
                "KD": f"{kd_loss.item():.4f}",
                "StudentCE": f"{student_ce_loss.item():.4f}"
            })

        try:
            train_auc = roc_auc_score(all_labels, all_probs)
        except:
            train_auc = float('nan')

        epoch_time = time.time() - start_epoch

        val_acc, val_auc = evaluate_model_on_validation(student, val_loader, device)

        print(f"[Epoch {epoch+1}] "
              f"KD Loss: {total_kd_loss / len(train_loader):.4f} | "
              f"Train AUC: {train_auc:.4f} | "
              f"Val AUC: {val_auc:.4f} | Val ACC: {val_acc:.4f} | "
              f"Time: {epoch_time:.2f}s")

    total_time = time.time() - start_training
    def count_all_parameters(model):
        return sum(p.numel() for p in model.parameters())

    def count_trainable_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Số tham số (Trainable) của Student: {count_trainable_parameters(student):,}")
    print(f"Tổng số tham số của Student: {count_all_parameters(student):,}")

    print(f"Số tham số (Trainable) của Teacher: {count_trainable_parameters(teacher):,}")
    print(f"Tổng số tham số của Teacher: {count_all_parameters(teacher):,}")

    print(f"⏱️ Tổng thời gian training: {total_time:.2f} giây ({total_time/60:.2f} phút)")


In [23]:
from sklearn.metrics import classification_report
import torch

def evaluate_models_report(student, teacher, dataloader, device):
    student.eval()
    teacher.eval()

    all_labels = []
    student_preds = []
    teacher_preds = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            s_logits = student(x)
            t_logits = teacher(x)

            student_cls = torch.argmax(s_logits, dim=1).cpu().numpy()
            teacher_cls = torch.argmax(t_logits, dim=1).cpu().numpy()
            true_labels = y.cpu().numpy()

            student_preds.extend(student_cls)
            teacher_preds.extend(teacher_cls)
            all_labels.extend(true_labels)

    print("📘 [Student Model] Classification Report:")
    print(classification_report(all_labels, student_preds, digits=4))

    print("📗 [Teacher Model] Classification Report:")
    print(classification_report(all_labels, teacher_preds, digits=4))


In [24]:
train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10)

[Epoch 1] KD Loss: 0.2609 | Train AUC: 0.8541 | Val AUC: 0.8867 | Val ACC: 0.7190 | Time: 426.38s


[Epoch 2] KD Loss: 0.2279 | Train AUC: 0.8984 | Val AUC: 0.8866 | Val ACC: 0.7157 | Time: 427.04s


[Epoch 3] KD Loss: 0.2184 | Train AUC: 0.9038 | Val AUC: 0.8709 | Val ACC: 0.7202 | Time: 426.49s


[Epoch 4] KD Loss: 0.2125 | Train AUC: 0.9047 | Val AUC: 0.8948 | Val ACC: 0.7259 | Time: 427.16s


[Epoch 5] KD Loss: 0.2083 | Train AUC: 0.9049 | Val AUC: 0.8853 | Val ACC: 0.7410 | Time: 428.04s


[Epoch 6] KD Loss: 0.2055 | Train AUC: 0.9045 | Val AUC: 0.8784 | Val ACC: 0.7279 | Time: 428.64s


[Epoch 7] KD Loss: 0.2038 | Train AUC: 0.9038 | Val AUC: 0.8739 | Val ACC: 0.7403 | Time: 428.89s


[Epoch 8] KD Loss: 0.2025 | Train AUC: 0.9041 | Val AUC: 0.8732 | Val ACC: 0.7324 | Time: 428.33s


[Epoch 9] KD Loss: 0.2015 | Train AUC: 0.9043 | Val AUC: 0.8847 | Val ACC: 0.7021 | Time: 429.05s


[Epoch 10] KD Loss: 0.2007 | Train AUC: 0.9043 | Val AUC: 0.8987 | Val ACC: 0.7133 | Time: 428.34s
Số tham số (Trainable) của Student: 11,177,538
Tổng số tham số của Student: 11,177,538
Số tham số (Trainable) của Teacher: 23,512,130
Tổng số tham số của Teacher: 23,512,130
⏱️ Tổng thời gian training: 5037.23 giây (83.95 phút)


In [25]:
evaluate_models_report(student, teacher, test_loader, device)

📘 [Student Model] Classification Report:
              precision    recall  f1-score   support

           0     0.9286    0.4627    0.6177     20000
           1     0.6422    0.9644    0.7710     20000

    accuracy                         0.7136     40000
   macro avg     0.7854    0.7136    0.6943     40000
weighted avg     0.7854    0.7136    0.6943     40000

📗 [Teacher Model] Classification Report:
              precision    recall  f1-score   support

           0     0.7202    0.3108    0.4342     20000
           1     0.5606    0.8792    0.6846     20000

    accuracy                         0.5950     40000
   macro avg     0.6404    0.5950    0.5594     40000
weighted avg     0.6404    0.5950    0.5594     40000



In [26]:
torch.save(student.state_dict(), "/kaggle/working/vgg11_student.pth")